# 08 — Failures and Retries

**Related guide:** [08_FAILURES_AND_RETRIES.md](../docs/guides/08_FAILURES_AND_RETRIES.md)

**Related script:** [08_failure.py](../examples/08_failure.py)

**Related script:** [09_retry.py](../examples/09_retry.py)

This notebook follows the DX05 loop: **Concept → Construction → Inspection → Modification → Execution → Observation → Experiment**.

## Construction

Create a deterministic transient failure and let the retry policy recover.

In [ ]:
from pyworkflowkit import (
    BackoffStrategy,
    RetryPolicy,
    TaskHandle,
    WorkflowRuntime,
    task,
    workflow,
)

calls = 0


@task(
    retry_policy=RetryPolicy(
        max_attempts=2,
        backoff_strategy=BackoffStrategy.NONE,
        retryable_error_categories=frozenset({"RuntimeError"}),
    )
)
def unstable() -> str:
    global calls
    calls += 1
    if calls == 1:
        raise RuntimeError("temporary notebook failure")
    return "recovered"


@workflow(id="notebooks.retry", version="1")
def demo() -> tuple[TaskHandle, ...]:
    return (unstable,)

In [ ]:
runtime = WorkflowRuntime()
runtime.register(unstable.handler_ref, unstable.handler)
run = runtime.run(demo.build())
event_types = [event.event_type.value for event in runtime.events(run.run_id)]
print({"status": run.status.value, "attempts": calls, "retry": "TASK_RETRYING" in event_types})

## Experiment

Change `max_attempts` to `1` and predict the terminal outcome before executing.